In [ ]:
import json, os, random, time
from datetime import datetime

SAVE_FILE = "progress.json"
LEADERBOARD_FILE = "leaderboard.json"
TOP_N = 10

BANNER = r"""
   ____      _               _   _            _
  / ___|   _| |__   ___ _ __| | | | __ _  ___| | _____ _ __
 | |  | | | | '_ \ / _ \ '__| |_| |/ _` |/ __| |/ / _ \ '__|
 | |__| |_| | |_) |  __/ |  |  _  | (_| | (__|   <  __/ |
  \____\__, |_.__/ \___|_|  |_| |_|\__,_|\___|_|\_\___|_|
       |___/
"""

TERMINAL = r"""
+--------------------------------------------------+
|  [::] CYBER LAB TERMINAL v3.7                    |
|  user@colab:~$                                   |
+--------------------------------------------------+
"""

ESCAPED_BANNER = r"""
███████╗███████╗ ██████╗ █████╗ ██████╗ ███████╗██████╗
██╔════╝██╔════╝██╔════╝██╔══██╗██╔══██╗██╔════╝██╔══██╗
█████╗  ███████╗██║     ███████║██████╔╝█████╗  ██║  ██║
██╔══╝  ╚════██║██║     ██╔══██║██╔═══╝ ██╔══╝  ██║  ██║
███████╗███████║╚██████╗██║  ██║██║     ███████╗██████╔╝
╚══════╝╚══════╝ ╚═════╝╚═╝  ╚═╝╚═╝     ╚══════╝╚═════╝
"""

ROOM_HEADERS = {
    1: r"""
   ______ _                          _ _
  |  ____(_)                        | | |
  | |__   _ _ __ _____      ___ __  | | |
  |  __| | | '__/ _ \ \ /\ / / '_ \ | | |
  | |    | | | |  __/\ V  V /| | | ||_|_|
  |_|    |_|_|  \___| \_/\_/ |_| |_|(_|_)
""",
    2: r"""
  _____                           _   _
 |  __ \                         | | (_)
 | |__) |___  ___ ___  _ __   ___| |_ _  ___ ___
 |  _  // _ \/ __/ _ \| '_ \ / _ \ __| |/ __/ _ \
 | | \ \  __/ (_| (_) | | | |  __/ |_| | (_|  __/
 |_|  \_\___|\___\___/|_| |_|\___|\__|_|\___\___|
""",
    3: r"""
   ____  _             _   ____
  / __ \| |           | | |  _ \
 | |  | | |_ __   ___ | |_| |_) | ___ _ __   ___ _ __
 | |  | | | '_ \ / _ \| __|  _ < / _ \ '_ \ / _ \ '__|
 | |__| | | |_) | (_) | |_| |_) |  __/ | | |  __/ |
  \____/|_| .__/ \___/ \__|____/ \___|_| |_|\___|_|
          | |
          |_|
"""
}


DIFFICULTIES = {
    "easy":   {"attempts": 4, "hint_cost": 1, "wrong_cost": 1, "lockout_cost": 3, "timeout_cost": 2, "otp_wrong_cost": 1, "win_bonus": 15, "speed": 0.015},
    "normal": {"attempts": 3, "hint_cost": 2, "wrong_cost": 1, "lockout_cost": 5, "timeout_cost": 4, "otp_wrong_cost": 2, "win_bonus": 20, "speed": 0.020},
    "hard":   {"attempts": 2, "hint_cost": 3, "wrong_cost": 2, "lockout_cost": 7, "timeout_cost": 6, "otp_wrong_cost": 3, "win_bonus": 30, "speed": 0.020},
}


def clear():
    print("\n" * 35)

def divider():
    print("\n" + "-"*52 + "\n")

def loading(msg="Connecting", steps=18, speed=0.02):
    print()
    for i in range(steps + 1):
        done = "#" * i
        left = "-" * (steps - i)
        print(f"\r{msg}... [{done}{left}] {int((i/steps)*100)}%", end="")
        time.sleep(speed)
    print("\n")

def cyber_tip():
    tips = [
        "Tip: 'hint' costs points — use it smartly 😼",
        "Tip: Save anytime with 'save' so Colab reset won't hurt.",
        "Tip: Fewer mistakes = higher score.",
        "Tip: Difficulty affects attempts + hint cost.",
        "Tip: Leaderboard exists. Go for top score 🏆"
    ]
    print("💡 " + random.choice(tips))

def fun_fail():
    lines = [
        "Trace detected… just kidding 😭 (but try again)",
        "Skill issue detected 🤖 (nah, you got this)",
        "Firewall laughed at you. Rude.",
        "You got ratio’d by the terminal 💀",
        "Your keyboard lagged. Totally not your fault 😌"
    ]
    print("😂 " + random.choice(lines))

def mission_status(state):
    room = state["room"]
    done1 = "✔" if room >= 2 else "✘"
    done2 = "✔" if room >= 3 else "✘"
    done3 = "✔" if state.get("finished", False) else "✘"
    diff = state.get("difficulty", "normal").upper()
    print(f"MISSION STATUS  |  Difficulty: {diff}")
    print(f"[1] Firewall   {done1}")
    print(f"[2] Decrypt    {done2}")
    print(f"[3] Core OTP   {done3}")

def splash(state):
    clear()
    print(BANNER)
    print(TERMINAL)
    inv = ", ".join(state["inventory"]) if state["inventory"] else "Empty"
    print(f"👾 Agent: {state['player']}  |  🔐 Room: {state['room']}  |  ⭐ Score: {state['score']}")
    print(f"🎒 Inventory: {inv}")
    divider()
    mission_status(state)
    divider()
    cyber_tip()

def print_commands():
    print("\n🧠 Commands: hint | inv | save | lb | quit")
    print("   (Type these anytime at a prompt)\n")

def ask(prompt, valid=None):
    while True:
        ans = input(prompt).strip().lower()
        if valid is None or ans in valid:
            return ans
        print(f"❌ Invalid. Choose from: {', '.join(valid)}")

def safe_int(s):
    try:
        return int(s)
    except:
        return None


def new_game():
    return {
        "player": "",
        "room": 1,
        "inventory": [],
        "score": 0,
        "hints_used": 0,
        "difficulty": "normal",
        "started_at": datetime.now().isoformat(timespec="seconds"),
        "finished": False
    }

def save_game(state):
    with open(SAVE_FILE, "w") as f:
        json.dump(state, f, indent=2)
    print("✅ Progress saved!")

def load_game():
    if not os.path.exists(SAVE_FILE):
        return None
    with open(SAVE_FILE, "r") as f:
        return json.load(f)

def add_item(state, item):
    if item not in state["inventory"]:
        state["inventory"].append(item)
        print(f"🎒 You got: {item}")

def show_status(state):
    splash(state)


def load_leaderboard():
    if not os.path.exists(LEADERBOARD_FILE):
        return []
    try:
        with open(LEADERBOARD_FILE, "r") as f:
            data = json.load(f)
        return data if isinstance(data, list) else []
    except:
        return []

def save_leaderboard(entries):
    with open(LEADERBOARD_FILE, "w") as f:
        json.dump(entries, f, indent=2)

def update_leaderboard(player, score, difficulty):
    entries = load_leaderboard()
    entries.append({
        "player": player,
        "score": score,
        "difficulty": difficulty,
        "time": datetime.now().isoformat(timespec="seconds")
    })
    entries.sort(key=lambda x: (-x["score"], x["time"]))
    entries = entries[:TOP_N]
    save_leaderboard(entries)
    return entries

def print_leaderboard():
    entries = load_leaderboard()
    divider()
    print("🏆 LEADERBOARD (Top Scores)")
    if not entries:
        print("No scores yet. Be the first legend 😼")
        return
    for i, e in enumerate(entries, start=1):
        print(f"{i:>2}. {e['player']} — {e['score']} pts — {e.get('difficulty','?').upper()}  ({e['time']})")


def award(state, points):
    state["score"] += points
    print(f"✅ +{points} points! (Score: {state['score']})")

def penalty(state, points):
    state["score"] = max(0, state["score"] - points)
    print(f"⚠️ -{points} points. (Score: {state['score']})")

def use_hint(state, hints):
    prof = DIFFICULTIES[state["difficulty"]]
    if state["hints_used"] >= 10:
        print("🚫 Hint limit reached for this run.")
        return
    state["hints_used"] += 1
    idx = min(state["hints_used"] - 1, len(hints) - 1)
    print(f"💡 HINT ({state['hints_used']}): {hints[idx]}")
    penalty(state, prof["hint_cost"])

def handle_common_commands(cmd, state):
    cmd = cmd.strip().lower()
    if cmd == "inv":
        show_status(state)
        return True
    if cmd == "save":
        save_game(state)
        return True
    if cmd == "lb":
        print_leaderboard()
        return True
    if cmd == "quit":
        save_game(state)
        print("👋 Exiting... Progress saved.")
        raise SystemExit
    return False

def caesar_encrypt(text, shift):
    out = []
    for ch in text:
        if ch.isalpha():
            base = ord('a') if ch.islower() else ord('A')
            out.append(chr((ord(ch) - base + shift) % 26 + base))
        else:
            out.append(ch)
    return "".join(out)

def init_puzzles(state):
    if isinstance(state.get("puzzles"), dict):
        return

    seed = random.randint(100000, 999999)
    rng = random.Random(seed)

    N = rng.randint(3, 7)
    M = rng.randint(1, 12)
    room1_pass = (2 ** N) + M

    K = rng.randint(4, 12)
    messages = [
        f"the key is {K}",
        f"key={K} dont leak",
        f"core key -> {K}",
        f"use key {K} asap"
    ]
    room2_plain = rng.choice(messages)
    room2_cipher = caesar_encrypt(room2_plain, N)

    state["puzzles"] = {
        "seed": seed,
        "room1": {"N": N, "M": M, "user": "root", "pass": str(room1_pass)},
        "room2": {"shift": N, "plain": room2_plain, "cipher": room2_cipher, "key_num": K},
        "room3": {"key_num": K}
    }


def room1_firewall(state):
    prof = DIFFICULTIES[state["difficulty"]]
    show_status(state)
    print(ROOM_HEADERS[1])
    loading("Booting firewall console", speed=prof["speed"])

    pz = state["puzzles"]["room1"]
    N, M = pz["N"], pz["M"]

    print("🖥️ ROOM 1: Firewall Gateway")
    print("You jack into a secured terminal. A firewall blocks the network tunnel.")
    print("Sticky note:")
    print("  'USER: Linux superuser account name'")
    print("Another note:")
    print(f"  'PASS: (2^{N}) + {M}'")
    divider()

    hints = [
        "Linux superuser is typically 'root'.",
        f"Compute 2^{N} then add {M}.",
        f"2^{N} = {2**N}, password = {2**N}+{M} = {pz['pass']}."
    ]

    attempts = prof["attempts"]
    while attempts > 0:
        print_commands()
        u = input("Enter username >>> ").strip().lower()
        if handle_common_commands(u, state): continue
        if u == "hint":
            use_hint(state, hints); continue

        p = input("Enter password >>> ").strip().lower()
        if handle_common_commands(p, state): continue
        if p == "hint":
            use_hint(state, hints); continue

        if u == pz["user"] and p == pz["pass"]:
            loading("Bypassing firewall rules", speed=prof["speed"])
            print("✅ Access granted. Tunnel opened.")
            add_item(state, "access_token")
            award(state, 10 + attempts)
            state["room"] = 2
            save_game(state)
            return
        else:
            attempts -= 1
            print(f"❌ Access denied. Attempts left: {attempts}")
            fun_fail()
            penalty(state, prof["wrong_cost"])

    print("\n🚨 LOCKOUT! Alarm triggered.")
    print("Backup port found... you can retry later.")
    penalty(state, prof["lockout_cost"])


def caesar_decrypt(text, shift):
    out = []
    for ch in text:
        if ch.isalpha():
            base = ord('a') if ch.islower() else ord('A')
            out.append(chr((ord(ch) - base - shift) % 26 + base))
        else:
            out.append(ch)
    return "".join(out)

def normalize_text(s):

    return " ".join(s.strip().lower().split())

def room2_decrypt(state):
    prof = DIFFICULTIES[state["difficulty"]]
    show_status(state)
    print(ROOM_HEADERS[2])

    if "access_token" not in state["inventory"]:
        print("🚫 You need access_token from Room 1.")
        return

    loading("Intercepting broadcast", speed=prof["speed"])

    pz = state["puzzles"]["room2"]
    cipher = pz["cipher"]
    shift = pz["shift"]
    correct_plain = normalize_text(pz["plain"])

    print("📡 ROOM 2: Encrypted Relay Node")
    print(f"Ciphertext: {cipher}")
    print("Clue:")
    print("  'Shift equals the exponent N used in Room 1.'")
    divider()

    hints = [
        "Room 1 used (2^N)+M. That N is the Caesar shift here.",
        f"Shift = {shift}. Decrypt by shifting BACK.",
        f"Correct plaintext (normalized) = '{correct_plain}'."
    ]

    attempts = prof["attempts"]
    while attempts > 0:
        print_commands()
        ans = input("Type decrypted message >>> ").strip().lower()
        if handle_common_commands(ans, state): continue
        if ans == "hint":
            use_hint(state, hints); continue

        if normalize_text(ans) == correct_plain:
            loading("Unlocking relay node", speed=prof["speed"])
            print("✅ Decryption successful.")
            print(f"Plaintext: '{pz['plain']}'")
            add_item(state, "decryption_key")
            add_item(state, f"key_{pz['key_num']}")
            award(state, 12 + attempts)
            state["room"] = 3
            save_game(state)
            return
        else:
            attempts -= 1
            print(f"❌ Wrong plaintext. Attempts left: {attempts}")
            fun_fail()
            penalty(state, prof["wrong_cost"])

    print("\n⚠️ Relay node timed out. Try again later.")
    penalty(state, prof["timeout_cost"])

def sum_digits(n):
    return sum(int(d) for d in str(abs(int(n))))

def room3_otp(state):
    prof = DIFFICULTIES[state["difficulty"]]
    show_status(state)
    print(ROOM_HEADERS[3])

    if "decryption_key" not in state["inventory"]:
        print("🚫 You need decryption_key from Room 2.")
        return False

    loading("Connecting to core server", speed=prof["speed"])

    key_num = state["puzzles"]["room3"]["key_num"]

    print("🏁 ROOM 3: Core Server Override")
    print("OTP rule:")
    print("  OTP = (sum of digits of your score) + (key number)")
    print(f"  key number = {key_num}")
    divider()

    hints = [
        "Sum the digits of your current score.",
        f"Add the key ({key_num}).",
        "Example: score 27 -> 2+7=9 -> OTP = 9 + key."
    ]

    attempts = prof["attempts"]
    while attempts > 0:
        print_commands()
        otp = input("Enter OTP >>> ").strip().lower()
        if handle_common_commands(otp, state): continue
        if otp == "hint":
            use_hint(state, hints); continue

        correct_otp = sum_digits(state["score"]) + key_num
        val = safe_int(otp)
        if val is not None and val == correct_otp:
            loading("Executing shutdown", speed=prof["speed"])
            print("✅ OTP accepted. Server override complete.")
            award(state, prof["win_bonus"])
            divider()
            print(ESCAPED_BANNER)
            print("🎉 YOU ESCAPED THE CYBER LAB!")
            print(f"🏁 Final Score: {state['score']}")
            state["finished"] = True
            return True
        else:
            attempts -= 1
            print(f"❌ Invalid OTP. Attempts left: {attempts}")
            fun_fail()
            penalty(state, prof["otp_wrong_cost"])

    print("\n🚨 Too many failed OTP attempts. System rebooted.")
    penalty(state, prof["lockout_cost"])
    return False

def play(state):
    prof = DIFFICULTIES[state["difficulty"]]
    show_status(state)
    loading("Initializing mission", speed=prof["speed"])

    print("🕶️ CYBER/HACKER ESCAPE: Mission Started")
    print("Break 3 layers and escape before traces catch you.\n")
    divider()

    while True:
        if state["room"] == 1:
            room1_firewall(state)
        elif state["room"] == 2:
            room2_decrypt(state)
        elif state["room"] == 3:
            finished = room3_otp(state)
            if finished:
                save_game(state)
                update_leaderboard(state["player"], state["score"], state["difficulty"])
                print_leaderboard()
                divider()
                print("✅ Score saved to leaderboard!")
                break
        else:
            print("⚠️ Unknown state. Resetting to Room 1.")
            state["room"] = 1

print_leaderboard()

state = load_game()
if state:
    print("\n💾 Save file found!")
    c = ask("Type 'c' to continue or 'n' for new game: ", valid={"c","n"})
    if c == "n":
        state = new_game()
else:
    state = new_game()

if not state["player"]:
    state["player"] = input("Enter your name: ").strip() or "Player"

if state.get("room", 1) == 1 and not state.get("puzzles"):
    print("\nSelect difficulty: easy / normal / hard")
    d = ask("Difficulty >>> ", valid=set(DIFFICULTIES.keys()))
    state["difficulty"] = d

init_puzzles(state)
save_game(state)

play(state)


----------------------------------------------------

🏆 LEADERBOARD (Top Scores)
No scores yet. Be the first legend 😼
Enter your name: Aizufatu

Select difficulty: easy / normal / hard
Difficulty >>> easy
✅ Progress saved!





































   ____      _               _   _            _
  / ___|   _| |__   ___ _ __| | | | __ _  ___| | _____ _ __
 | |  | | | | '_ \ / _ \ '__| |_| |/ _` |/ __| |/ / _ \ '__|
 | |__| |_| | |_) |  __/ |  |  _  | (_| | (__|   <  __/ |
  \____\__, |_.__/ \___|_|  |_| |_|\__,_|\___|_|\_\___|_|
       |___/


+--------------------------------------------------+
|  [::] CYBER LAB TERMINAL v3.7                    |
|  user@colab:~$                                   |
+--------------------------------------------------+

👾 Agent: Aizufatu  |  🔐 Room: 1  |  ⭐ Score: 0
🎒 Inventory: Empty

----------------------------------------------------

MISSION STATUS  |  Difficulty: EASY
[1] Firewall   ✘
[2] Decrypt    ✘
[3] Core OTP   ✘

-----------------

SystemExit: 

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
